# News Articles Grouping Research

The central hypothesis of this research is that by combining multiple layers of processing and analysis—rather than relying solely on semantic or topical similarity—we can more accurately group news articles into meaningful real-world events.

In other words, we do not want to cluster articles together simply because they mention the same high-level entity (e.g., the President of Argentina). Instead, we aim to group them based on the specific event that occurred. For example:

- *The Argentine president approves a set of laws preventing X, Y, and Z.*
- *The Argentine president vetoes a tax reform proposal.*

Although both articles mention the same public figure, they describe different real-world events and therefore should belong to different clusters.

---

## Baseline

As a baseline, we preprocess the articles by generating embeddings using `multilingual-e5-large`. This model was selected because it has been trained on diverse multilingual data, including news-style corpora, making it well-suited for capturing the semantic structure of journalistic text. Additionally, it supports both English and Spanish,the two languages evaluated in this experiment, allowing us to maintain a unified embedding space across languages.

In cases where an article exceeds the model's maximum context length, we truncate it rather than average multiple chunks. Averaging chunk embeddings may dilute event-specific signals and introduce smoothing effects that blur important distinctions. By applying a head-based truncation strategy with 512 tokens, we aim to preserve the most informative portion of the article, as news writing typically presents the core event details in the opening paragraphs.

Future experiments will explore alternative embedding models to assess their impact on clustering performance. However, `multilingual-e5-large` provides a strong and appropriate baseline for this task.

After embedding, we apply **HDBSCAN**, a density-based clustering algorithm that:

- Identifies clusters of varying density.
- Does not require specifying the number of clusters in advance.
- Labels outliers as noise.

| Metric | Score |
|------|------|
| Homogeneity Score | 0.5550 |
| Completeness Score | 0.8228 |
| V-Measure Score | 0.6629 |
| Intra-cluster Similarity | 0.8869 |


HDBSCAN groups articles based on proximity in the embedding space. While it performs well for clearly separated events, in many cases it clusters articles according to broad semantic similarity or shared topics rather than the specific real-world event being described. This limitation highlights the challenges of relying solely on embedding-based semantic clustering for event-level grouping. But this experiment provides a solid baseline and reference point from which we can analyze the limitations of purely semantic clustering and continue improving the pipeline in subsequent stages.

---

## Research Structure

### 1. Stage 1 - Article Pair Generation

- **Article preprocessing:**  
  We embed all articles using `multilingual-e5-large` with a 1024-dimensional representation. This provides a dense numerical representation of each article.

- **Similarity search algorithms:**  
  We compare two common nearest-neighbor search approaches:

  - **k-Nearest Neighbors (KNN):**  
    Focuses on exact similarity search, prioritizing precision at the expense of computational speed.

  - **Approximate Nearest Neighbors (ANN):**  
    Trades a small amount of accuracy for significantly improved speed and scalability, retrieving results that are "close enough."

  We will evaluate both approaches to determine which provides the best trade-off between efficiency and retrieval quality for our pipeline.

- **Pair construction:**  
  For each article, we retrieve its top *k* most similar articles (e.g., 10-20). We then construct pairs of the form:  
  (A, A₁), (A, A₂), ..., (A, Aₖ).  
  These candidate pairs will serve as inputs for deeper event-level comparison.

<br/>

### 2. Stage 2 - Pair Comparison

The objective of this stage is to determine whether two articles within a candidate pair describe the same real-world event. We use two complementary methods:

- **Cross-Encoder Classification Model:**  
  A custom fine-tuned transformer that takes both articles jointly as input and outputs a probability indicating whether they refer to the same event. Unlike bi-encoder embeddings, a cross-encoder evaluates the interaction between the two texts directly, enabling finer-grained comparison.

- **Entity Extraction and Analysis:**  
  This step includes:
  1. Named Entity Recognition (NER) to extract entities.
  2. Entity normalization using a knowledge base (Wikidata) to resolve aliases and canonicalize references.
  3. Jaccard similarity between the two normalized entity sets to compute an entity-overlap score.

Each method produces a similarity score. We then compute a weighted combination of these scores to obtain a final event-level similarity score for each article pair.

<br/>

### 3. Graph Generation

Once similarity scores are computed for all candidate pairs, we construct a graph:

- Each node represents an article.
- An edge between two nodes represents their final similarity score.
- The graph is sparse, as each article is only connected to its top-*k* nearest neighbors.
- Weak edges (below a predefined threshold) are removed to reduce noise.

We then apply and compare two graph-based grouping algorithms:

- **Louvain:**  
  A community detection algorithm that optimizes modularity and is well-suited for weighted graphs.

- **Connected Components:**  
  A simpler approach that groups nodes based solely on connectivity.

By comparing these methods, we aim to determine which produces more coherent event-level clusters for our task. The best-performing configuration will be adopted in the final research pipeline.

### Import Dependencies

In [ ]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 43.6 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [ ]:
# Paths for embeddings generation
DATASET_PATH = "/content/drive/MyDrive/Research/articles_grouping/articles_events.csv"
EMBEDDINGS_OUTPUT_PATH = "/content/drive/MyDrive/Research/articles_grouping/embeddings.dat"
# Paths for embeddings loading
DRIVE_PATH = "/content/drive/MyDrive/Research/articles_grouping/embeddings.dat"
LOCAL_PATH = "/content/local_embeddings.dat"

In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-large"

## 1.1 Data Preparation

In this section, we generate embeddings for approximately ~600k news articles using a combination of each article's **title and content** as input. This approach provides a richer contextual representation than using either component independently.

For embedding generation, we use the `sentence_transformers` library within a Google Colab environment. If the experiment is replicated locally, the same procedure can be executed using `Ollama`, provided the model is properly installed and configured.

As introduced earlier, we use `multilingual-e5-large`, a multilingual embedding model that supports both English and Spanish and is trained to produce high-quality semantic representations across languages. Its training data includes web and news-style corpora, making it particularly suitable for journalistic text.

To ensure efficient memory management, we implement a batching strategy during embedding generation. The computed embeddings are stored on disk, allowing us to generate them once and reuse them across multiple experiments without recomputation. This guarantees both computational efficiency and experimental consistency.

In [ ]:
# Setup Google Drive in Colab
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


#### Load Dataset

All articles should have values based on the data analysis notebook results but just in case we add the extra check.

In [ ]:
df = pd.read_csv(filepath_or_buffer=DATASET_PATH)
df = df.reset_index(drop=True)

df["title"] = df["title"].fillna("")
df["content"] = df["content"].fillna("")
df["text"] = "passage: " + df["title"] + "\n" + df["content"] # How the embedder needs to receive the corpus

print(f"Dataset loaded and formatted successfully with {df.shape[0]} examples")

Dataset loaded and formatted successfully with 647442 examples


#### Load Embedder Model

In [ ]:
emb_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

In [ ]:
TOTAL = len(df)
EMBEDDING_DIM = 1024
BATCH_SIZE = 2048

In [ ]:
embeddings_mmap = np.memmap(EMBEDDINGS_OUTPUT_PATH, dtype="float32", mode="w+", shape=(TOTAL, EMBEDDING_DIM))

for i in tqdm(range(0, TOTAL, BATCH_SIZE)):
    batch = df["text"].iloc[i : i + BATCH_SIZE].tolist()
    embeddings_mmap[i : i + BATCH_SIZE] = emb_model.encode(
        batch,
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    if i % (BATCH_SIZE * 10) == 0:
        embeddings_mmap.flush()

del embeddings_mmap

100%|██████████| 317/317 [2:54:51<00:00, 33.10s/it]


To efficiently utilize the available computational resources during embedding generation, we implemented a two-level batching strategy:

- **Data loading batch size:** 2,048 articles per batch.  
  This prevents loading the entire dataset into memory at once, reducing RAM usage and ensuring stability during processing.

- **Model inference batch size:** 256 articles per forward pass.  
  This value was selected to maximize GPU utilization while avoiding out-of-memory (OOM) errors.

This stage of the experiment was executed in a Google Colab Pro environment using an **NVIDIA A100 GPU with 40GB of VRAM**.  

Processing the full dataset of **647,442 articles** required **2 hours, 54 minutes, and 51 seconds**, including batching and disk storage of embeddings.

### Load Embeddings

> *Note: The embeddings were precomputed and stored on disk. If the embedding generation step has already been executed in the current session, the following cell can be skipped.*

In [ ]:
embeddings = np.fromfile(DRIVE_PATH, dtype="float32").reshape(TOTAL, EMBEDDING_DIM)

In [ ]:
print(f"Embeddings loaded with shape: {embeddings.shape}")
print(f"Total number of articles (rows): {embeddings.shape[0]}")
print(f"Embedding dimension (columns): {embeddings.shape[1]}")

Embeddings loaded with shape: (647442, 1024)
Total number of articles (rows): 647442
Embedding dimension (columns): 1024


## 1.2 Similarity Search Algorithm

In this section, we evaluate and compare **exact k-Nearest Neighbors (KNN)** and **Approximate Nearest Neighbors (ANN)** based on both accuracy and computational efficiency. The goal is to determine which method is better suited for large-scale similarity search over ~600k news article embeddings.

Given the size of the dataset, exact KNN can become computationally expensive, especially when computing similarities across the full embedding space. While it provides precise nearest neighbors, its scalability may be limited as the dataset grows.

ANN methods, on the other hand, trade a small amount of retrieval accuracy for significantly improved speed and scalability. These methods aim to retrieve neighbors that are sufficiently close to the true nearest neighbors, making them practical for large embedding collections.

We compare both approaches in terms of:

- **Retrieval quality** (e.g., recall@k or overlap with exact neighbors)
- **Query latency and indexing time**
- **Scalability with increasing dataset size**

Based on this evaluation, we will select the most appropriate similarity search strategy for the remainder of the research pipeline.

> *Note: For the implementation we will use Meta's library called **FAISS**, which has built-in implementations for both algorithms.*

In [ ]:
K_NEIGHBORS = 21 # One more to account for itself

### K-Nearest Neighbors (KNN)

In [ ]:
# Generate index intance in CPU
index_cpu = faiss.IndexFlatL2(EMBEDDING_DIM)
# Move index to GPU
res = faiss.StandardGpuResources()
index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
# Add all embeddings
index.add(embeddings)
print(f"Total vectors indexed: {index.ntotal}")

Total vectors indexed: 647442


In [ ]:
batch_size = 10_000
results = []

for i in tqdm(range(0, len(embeddings), batch_size)):
    batch = embeddings[i : i + batch_size]
    distances, indices = index.search(batch, k=K_NEIGHBORS)
    results.append(indices[:, 1:])

all_neighbors = np.concatenate(results, axis=0)
print(f"\nSearch complete. Total result sets: {len(results)}")

100%|██████████| 65/65 [02:26<00:00,  2.25s/it]


Search complete. Total result sets: 65


> *Note: KNN took 2 minutes and 26 seconds to find 20 neighbors for all 647,442 articles.*

### Approximate Nearest Neighbors (ANN)

In [ ]:
# Create quantizer for ANN search
quantizer = faiss.IndexFlatL2(EMBEDDING_DIM)
# Create ANN CPU instance
n_list = 1_000 # Amount of buckets to check
index_cpu = faiss.IndexIVFFlat(quantizer, EMBEDDING_DIM, n_list, faiss.METRIC_L2)
# Move ANN to GPU
res = faiss.StandardGpuResources()
index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
# Train index with all our embeddings
index.train(embeddings)
# Add embeddings to search
index.add(embeddings)
print(f"Total vectors indexed: {index.ntotal}")

Total vectors indexed: 647442


In [ ]:
index.nprobe = 10 # How many nearby clusters to visit
ann_results = []

for i in tqdm(range(0, len(embeddings), batch_size)):
    batch = embeddings[i : i + batch_size]
    distances, indices = index.search(batch, k=K_NEIGHBORS)
    ann_results.append(indices[:, 1:])

all_neighbors_ann = np.concatenate(ann_results, axis=0)
print(f"\nSearch complete. Total result sets: {len(ann_results)}")

100%|██████████| 65/65 [01:35<00:00,  1.48s/it]


Search complete. Total result sets: 65


> *Note: ANN took 1 minutes and 35 seconds to find 20 neighbors for all 647,442 articles.*

### Evaluation

In this subsection, we compare the results of exact k-Nearest Neighbors (KNN) with those of Approximate Nearest Neighbors (ANN).  

To evaluate retrieval quality, we compute the intersection between the neighbor sets returned by both methods over 200,000 query examples. This allows us to measure how closely the ANN results approximate the true nearest neighbors obtained via exact KNN.

The overlap between both neighbor sets serves as a proxy for recall and helps quantify the trade-off between speed and retrieval accuracy.

In [ ]:
eval_size = 200_000
found = 0

for i in range(eval_size):
    # Count how many of the true KNN neighbors were captured by ANN
    intersection = np.intersect1d(all_neighbors[i], all_neighbors_ann[i])
    found += len(intersection)

recall = found / (eval_size * (K_NEIGHBORS - 1))

print(f"Evaluation for {eval_size} samples:")
print(f"Final ANN Recall@k: {recall:.4f}")
print(f"KNN Time: 2 minutes and 26 seconds")
print(f"ANN Time: 1 minutes and 35 seconds")

Evaluation for 200000 samples:
Final ANN Recall@k: 0.9499
KNN Time: 2 minutes and 26 seconds
ANN Time: 1 minutes and 35 seconds


### Results

The results show that exact KNN required **2 minutes and 26 seconds** to retrieve the top-*k* similar articles for all query examples, whereas ANN completed the same task in **1 minute and 35 seconds**, representing approximately a **35% speed improvement**.

In terms of retrieval quality, the **Recall@k**, which measures the proportion of true nearest neighbors (as determined by exact KNN) that are successfully retrieved by ANN, is **0.9499 (~95%)**.  

This means that, on average, ANN retrieves roughly **19 out of every 20** true closest articles identified by exact KNN, indicating a strong trade-off between efficiency and accuracy.

<br/>

### Algorithm Conclusion

Given the substantial speed improvement (\~35%) and the high Recall@k (~95%), we select **ANN** as the similarity search method for the remainder of the research pipeline.

The small loss in exact neighbor retrieval is acceptable given the scale of the dataset (~600k articles) and the need for computational efficiency. Moreover, since this stage only generates candidate pairs that will later undergo deeper pairwise evaluation, near-perfect recall is sufficient for downstream processing.

### Articles Pair Generation

In [ ]:
# Create list of ids from the original dataframe
article_ids = df["id"].values

# Generate lists to then create the pairs
article_ids = article_ids.repeat(K_NEIGHBORS - 1)
neighbor_ids = article_ids[all_neighbors_ann.flatten()]

pairs_df = pd.DataFrame({
    "article_id": article_ids,
    "neighbor_id": neighbor_ids
})
print(f"Generated {len(pairs_df)} candidate pairs with original IDs")

Generated 12301398 candidate pairs with original IDs


> *Note: For each article, we generate 20 candidate pairs using the following structure:*  
> *(A, A₁), (A, A₂), ..., (A, Aₖ),*  
> *where each Aᵢ represents one of the top-*k* most similar articles retrieved during the similarity search stage.*

> *Total pairs: 12,301,398.*